# Word2Vec resources

### Alfio Ferrara


In [ ]:
from gensim.models import Word2Vec

## Main functionalities of `gensim` implementation

In [ ]:
from nltk.tokenize import word_tokenize
from scipy.spatial import distance
import pandas as pd
import ast
from nltk.tokenize import word_tokenize
from collections import defaultdict

In [ ]:
recipe_corpus = []
datafile = "/Users/Flint/Data/recipes/it_recipes.csv"
df = pd.read_csv(datafile, index_col=0)
df = df.dropna()
for i, row in df.iterrows():
    recipe = []
    category = row['Categoria'].lower()
    title = row['Nome'].lower()
    ingredients = [x[0].lower() for x in ast.literal_eval(row['Ingredienti'])]
    text = word_tokenize(row['Steps'].lower().replace("'", " "), language='italian')
    recipe.append(category)
    recipe.append(title)
    recipe.extend(ingredients)
    recipe.extend(text)
    recipe_corpus.append(recipe)

In [ ]:
categories = defaultdict(list)
for recipe in recipe_corpus:
    cat = recipe[0]
    categories[cat].append(recipe)
for cat, docs in categories.items():
    print(f"Num di ricette in {cat}: {len(docs)}")

In [ ]:
print(f"Esempio di dolci: {categories['dolci'][0][:6]}")
print(f"Esempio di primi: {categories['primi piatti'][0][:6]}")

In [ ]:
recipe_model = Word2Vec(sentences=recipe_corpus, vector_size=300, window=9, 
                        min_count=10, workers=8, epochs=50)

### Similarity

In [ ]:
recipe_model.wv.most_similar('bollire')

## Compositionality

In [ ]:
dm = recipe_model.wv.doesnt_match(['pasta', 'spaghetti', 'noodles', 'mela'])
common = recipe_model.wv.get_mean_vector(['pasta', 'spaghetti', 'noodles', 'risotto'])
common_word = recipe_model.wv.similar_by_vector(common)
analogy = recipe_model.wv.most_similar(positive=['bollire', 'olio'], negative=['acqua'])

In [ ]:
print(f"Doesn't match: {dm}")
print(f"Common terms: {common_word}")
print(f"Analogy: {analogy}")

## Compare models vectors to measure a shift in meaning

In [ ]:
dolci_corpus = categories['dolci']
primi_corpus = categories['primi piatti']
print(f"Dolci: {len(dolci_corpus)}, Primi: {len(primi_corpus)}")

### Fine tune the global model to specific sub-corpora

In [ ]:
import copy
import numpy as np

In [ ]:
generic_corpus = []
for cat, docs in categories.items():
    if cat not in ['primi piatti', 'dolci']:
        generic_corpus.extend(docs)
recipe_model = Word2Vec(sentences=generic_corpus, vector_size=300, window=9, 
                        min_count=10, workers=8, epochs=50)

In [ ]:
m_d = copy.deepcopy(recipe_model)
m_p = copy.deepcopy(recipe_model)

In [ ]:
m_d.train(dolci_corpus, total_examples=recipe_model.corpus_count, epochs=recipe_model.epochs)
m_p.train(primi_corpus, total_examples=recipe_model.corpus_count, epochs=recipe_model.epochs)

In [ ]:
word = 'zucchero'
general = dict(recipe_model.wv.most_similar(word))
dol = dict(m_d.wv.most_similar(word))
pri = dict(m_p.wv.most_similar(word))

In [ ]:
for k, v in general.items():
    g, d, p = v, dol.get(k, 0), pri.get(k, 0)
    print(f"{k} => {np.round(g, 3)} | D: {np.round(d - g, 3)} | P: {np.round(p - g, 3)}")

In [ ]:
word = 'zucchero'
v0, vit, vch = recipe_model.wv.get_vector(word), m_d.wv.get_vector(word), m_p.wv.get_vector(word)

In [ ]:
print(f"Moving to Dolci: {distance.cosine(vit, v0)}")
print(f"Moving to Primi: {distance.cosine(vch, v0)}")
print(f"Moving from Dolci to Primi: {distance.cosine(vch, vit)}")

## Allenamento di modelli indipendenti

In [ ]:
primi_model = Word2Vec(sentences=primi_corpus, vector_size=300, window=9, 
                        min_count=10, workers=8, epochs=50)
dolci_model = Word2Vec(sentences=dolci_corpus, vector_size=300, window=9, 
                        min_count=10, workers=8, epochs=50)

In [ ]:
word = "zucchero"
primi_model.wv.most_similar(word)

In [ ]:
dolci_model.wv.most_similar(word)